In [ ]:
# OLD MODELS

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv("Dataset/Features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Function to create specific targets (Will I do X in the next 30 mins?)
def get_target(df, activity_name):
    times = df[df['activity'] == activity_name]['timestamp'].sort_values()
    df[f'target_{activity_name}'] = df['timestamp'].apply(
        lambda x: 1 if any((t > x and (t - x).total_seconds()/60 <= 30) for t in times) else 0
    )
    return df[f'target_{activity_name}']

# Define activities and features
activities = ['drink', 'eat', 'outside_out']
features = ["hour", "day_of_week", "is_weekend", "minutes_since_midnight", 
            "time_since_last_drink", "time_since_last_eat", "time_since_last_outside"]

print("--- PERFORMANCE RESULTS ---")

for act in activities:
    y = get_target(df, act)
    X = df[features].fillna(0)
    
    # Split 80% training, 20% testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model.fit(X_train, y_train)
    
    print(f"\nModel: {act.upper()}")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

--- PERFORMANCE RESULTS ---

Model: DRINK
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        16
           1       0.75      1.00      0.86         3

    accuracy                           0.95        19
   macro avg       0.88      0.97      0.91        19
weighted avg       0.96      0.95      0.95        19


Model: EAT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19


Model: OUTSIDE_OUT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19



In [4]:
import pandas as pd

# Define your scenarios with the new '7-day average' columns
# Scenario 1: Monday - Drank 8 times today, average for the week is 7
# Scenario 2: Friday - Drank 6 times today, average for the week is 6.5
# Scenario 3: Saturday - High activity today (10), but average is lower (8)
# Scenario 4: Sunday - Low activity today (5), average is 7.5
data = {
    'day_of_week': [0, 4, 5, 6],
    'is_weekend': [0, 0, 1, 1],
    'drink': [8, 6, 10, 5],
    'eat': [3, 2, 4, 2],
    'drink_7day_avg': [7.0, 6.5, 8.0, 7.5],
    'eat_7day_avg': [3.0, 2.5, 3.5, 3.0]
}

# Create the DataFrame
df_test_daily = pd.DataFrame(data)

# Save it to the Dataset folder
import os
if not os.path.exists('Dataset'):
    os.makedirs('Dataset')

df_test_daily.to_csv("Dataset/Daily_Test_Data.csv", index=False)

print("Daily_Test_Data.csv successfully updated with 7-day averages!")
print(df_test_daily)

Daily_Test_Data.csv successfully updated with 7-day averages!
   day_of_week  is_weekend  drink  eat  drink_7day_avg  eat_7day_avg
0            0           0      8    3             7.0           3.0
1            4           0      6    2             6.5           2.5
2            5           1     10    4             8.0           3.5
3            6           1      5    2             7.5           3.0


In [ ]:
import pandas as pd
import joblib
import numpy as np

# Load the Real Data
try:
    df = pd.read_csv("Dataset/Daily_Features.csv")
except FileNotFoundError:
    print("Error: Daily_Features.csv not found.")
    exit()

# Prepare the Test Set (Last 20% of data)
cutoff = int(len(df) * 0.8)
test_data = df.iloc[cutoff:].copy()
features = ['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg']

# Load Model and Predict
print("--- TESTING DRINK MODEL ---")
model = joblib.load("daily_drink_forecast.roboai")
predictions = model.predict(test_data[features])

# Compare and Grade
test_data['actual'] = test_data['target_drink_tomorrow']
test_data['predicted'] = predictions
test_data['error'] = abs(test_data['actual'] - test_data['predicted'])

# Accuracy Formula: 100% - (Percentage Error)
# We clip at 0 to prevent negative scores if the guess is way off
test_data['accuracy'] = 100 * (1 - (test_data['error'] / test_data['actual']))
test_data['accuracy'] = test_data['accuracy'].clip(lower=0)

# Show Results
print(test_data[['date', 'actual', 'predicted', 'accuracy']].head(10))
print("\n" + "="*30)
print(f"OVERALL ACCURACY: {test_data['accuracy'].mean():.2f}%")
print(f"AVERAGE ERROR:    {test_data['error'].mean():.2f} drinks")
print("="*30)

--- TESTING DRINK MODEL ---
           date  actual  predicted   accuracy
181  2019-03-01     6.0      5.855  97.583333
182  2019-03-02     5.0      5.520  89.600000
183  2019-03-03     7.0      7.540  92.285714
184  2019-03-04    12.0     10.660  88.833333
185  2019-03-05     8.0      8.170  97.875000
186  2019-03-06     7.0      7.470  93.285714
187  2019-03-07     7.0      7.370  94.714286
188  2019-03-08     5.0      5.710  85.800000
189  2019-03-09     8.0      7.520  94.000000
190  2019-03-10    11.0     10.795  98.136364

OVERALL ACCURACY: 92.47%
AVERAGE ERROR:    0.45 drinks


In [ ]:
import pandas as pd
import joblib
import numpy as np

# Load the Real Data
try:
    df = pd.read_csv("Dataset/Daily_Features.csv")
except FileNotFoundError:
    print("Error: Daily_Features.csv not found.")
    exit()

# Prepare the Test Set (Last 20% of data)
cutoff = int(len(df) * 0.8)
test_data = df.iloc[cutoff:].copy()
features = ['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg']

# Load Model and Predict
print("--- TESTING EAT MODEL ---")
model = joblib.load("daily_eat_forecast.roboai")
predictions = model.predict(test_data[features])

# Compare and Grade
test_data['actual'] = test_data['target_eat_tomorrow']
test_data['predicted'] = predictions
test_data['error'] = abs(test_data['actual'] - test_data['predicted'])

# Accuracy Formula
test_data['accuracy'] = 100 * (1 - (test_data['error'] / test_data['actual']))
test_data['accuracy'] = test_data['accuracy'].clip(lower=0)

# Show Results
print(test_data[['date', 'actual', 'predicted', 'accuracy']].head(10))
print("\n" + "="*30)
print(f"OVERALL ACCURACY: {test_data['accuracy'].mean():.2f}%")
print(f"AVERAGE ERROR:    {test_data['error'].mean():.2f} meals")
print("="*30)

--- TESTING EAT MODEL ---
           date  actual  predicted   accuracy
181  2019-03-01     4.0   3.695000  92.375000
182  2019-03-02     4.0   3.640000  91.000000
183  2019-03-03     4.0   3.760000  94.000000
184  2019-03-04     4.0   4.150000  96.250000
185  2019-03-05     4.0   4.130000  96.750000
186  2019-03-06     5.0   4.420000  88.400000
187  2019-03-07     5.0   4.360000  87.200000
188  2019-03-08     2.0   2.240000  88.000000
189  2019-03-09     3.0   2.970000  99.000000
190  2019-03-10     4.0   4.106667  97.333333

OVERALL ACCURACY: 89.75%
AVERAGE ERROR:    0.28 meals
